<a href="https://colab.research.google.com/github/Siuumanth/Machine-Learning-and-other-notebooks/blob/main/AutoComplete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Guided Project from Neural nine

In [ ]:
!pip install tensorflow
!pip install nltk

In [ ]:
import tensorflow as tf
from nltk.tokenize import RegexpTokenizer


import numpy as np
import pandas as pd
import pickle

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Activation
from tensorflow.keras.optimizers import RMSprop
from tensorflow import keras

In [ ]:
data= pd.read_csv("/content/drive/MyDrive/datasets/auto complete/fake_or_real_news.csv")

In [ ]:
data.head()

In [ ]:
text = list(data.text.values)
joined_text= ' '.join(text)
#we are only extracting the text column and preprocess it for training

In [ ]:
partial_text = joined_text [:10000]  # first 1000 characters

Seperating the joined text into individual words(tokens) and storing them all in a list :

In [ ]:
tokenizer_ = RegexpTokenizer(r'\w+')
tokens = tokenizer_.tokenize(partial_text.lower())

In [ ]:
type(tokens)

list

Getting Unique tokens


In [ ]:
unique_tokens = np.unique(tokens)
unique_tokens_index = {token: idx for idx, token in enumerate(unique_tokens)} #assigning a unique number to each token

In [ ]:
len(tokens)

1753

In [ ]:
n_words = 10   # we are gonna see 10 words and predict the next sentance
input_words = []
next_words = []

#### So we are gonna look at 10 words and predict the next word , and then we look at the next 10 words and precict the next word

#### our training data is going to be this , from the words we have collected we are gonna make partitions based on the above idea and use it as our training data

In [ ]:
for i in range(len(tokens) - n_words):
    input_words.append(tokens[i:i + n_words])   #This contains all possible 10 words iin the dataset
    next_words.append(tokens[i + n_words])      #contains all predicition word(labels) for the input words


In [ ]:
print(input_words[0:5])
print(next_words[0:5])


## One hot encoding :


In [ ]:
X = np.zeros((len(input_words), n_words, len(unique_tokens)), dtype=bool)
y = np.zeros((len(next_words), len(unique_tokens)), dtype=bool)

len(input_words): Represents the number of samples or sequences in your dataset.

n_words: Represents the number of words (or tokens) in each sequence. This makes the model consider a fixed context window for predictions, where n_words is the size of the window.

len(unique_tokens): Represents the total number of unique tokens (vocabulary size). Each word in the sequence is one-hot encoded as a vector of this length.

In [ ]:
X.shape

(1743, 10, 694)

In [ ]:
for index, words in enumerate(input_words):
    for jndex, word in enumerate(words):
        X[index, jndex, unique_tokens_index[word]] = 1    #one hot encoding
        y[index, unique_tokens_index[next_words[index]]] = 1
# ONE HOT DONE

## Building the Model

return_sequences=True:
This ensures that the output of this LSTM layer is a sequence of outputs, one for each time step in the input sequence. This is necessary when stacking LSTM layers, as the next LSTM layer needs a sequence as its input.

Why No return_sequences=True Here?

This second LSTM layer is likely intended to produce a final summary representation of the sequence, which is passed on to the next layer. Therefore, only the final output (the last time step) is needed, not the entire sequence.

In [ ]:
model = Sequential()
model.add(LSTM(128, input_shape=(n_words, len(unique_tokens)), return_sequences=True))
model.add(LSTM(128))
model.add(Dense(len(unique_tokens)))
model.add(Activation('softmax'))

# OR

In [ ]:
model = keras.Sequential([
    keras.layers.LSTM(128, input_shape=(n_words, len(unique_tokens)), return_sequences=True),
    keras.layers.LSTM(128),
    keras.layers.Dense(len(unique_tokens),activation = 'softmax'),
])

In [ ]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_6 (LSTM)                        │ (None, 10, 128)             │         421,376 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_7 (LSTM)                        │ (None, 128)                 │         131,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 694)                 │          89,526 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 642,486 (2.45 MB)

 Trainable params: 642,486 (2.45 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer=RMSprop(learning_rate=0.01), metrics=['accuracy'])
model.fit(X, y, batch_size=128, epochs=30, shuffle=True)

Epoch 1/30
14/14 ━━━━━━━━━━━━━━━━━━━━ 7s 158ms/step - accuracy: 0.0380 - loss: 6.3206
Epoch 2/30
14/14 ━━━━━━━━━━━━━━━━━━━━ 3s 207ms/step - accuracy: 0.0550 - loss: 5.8133
Epoch 3/30
14/14 ━━━━━━━━━━━━━━━━━━━━ 3s 84ms/step - accuracy: 0.0577 - loss: 5.7781
Epoch 4/30
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 89ms/step - accuracy: 0.0583 - loss: 5.7754
Epoch 5/30
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 98ms/step - accuracy: 0.0658 - loss: 5.7326
Epoch 6/30
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 96ms/step - accuracy: 0.0599 - loss: 5.7254
Epoch 7/30
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 88ms/step - accuracy: 0.0560 - loss: 5.6755
Epoch 8/30
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 88ms/step - accuracy: 0.0623 - loss: 5.5659
Epoch 9/30
14/14 ━━━━━━━━━━━━━━━━━━━━ 3s 138ms/step - accuracy: 0.0674 - loss: 5.5067
Epoch 10/30
14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 147ms/step - accuracy: 0.0646 - loss: 5.3757
Epoch 11/30
14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 124ms/step - accuracy: 0.0821 - loss: 5.1801
Epoch 12/30
14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 86ms/step - accu

Now we we will learn to predict next word and text

In [ ]:
def predict_next_word(input_text, n_best):
    input_text = input_text.lower()
    X = np.zeros((1, n_words, len(unique_tokens)))
    for i,word in enumerate(input_text.split()):
      X[0,i,unique_tokens_index[word]] = 1
    predictions = model.predict(X)[0]
    return np.argpartition(predictions, -n_best)[-n_best:]

In [ ]:
possible = predict_next_word("The president ", 5)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step


In [ ]:
print([unique_tokens[idx] for idx in possible])

['risk', 'it', 's', 'the', 'an']


In [ ]:
def generate_text(input_text, text_length, creativity=3):
    word_sequence = input_text.split()
    current = 0
    for _ in range(text_length):
        sub_sequence = " ".join(tokenizer.tokenize(" ".join(word_sequence).lower())[current:current+n_words])
        sample_data = np.zeros((1, n_words, len(unique_tokens)))
